In [ ]:
# --- path configuration -------------------------------------------------
# Every path goes through resolve() (inputs under data/) or out_path() (this run's output).
# Set SURFACEOME_DATA to point at the data archive; see README.md.
from surfaceome_config import resolve, out_path


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from surfaceomeTopography import cell_surface, mean_height, plot_surfaces
%matplotlib inline

## Cell_Expression_E-PROT-1_human_proteome_map
**Figure 2** Immune Cell heights

In [ ]:


surfaceome = pd.read_csv(resolve('inputs/mapping/uniprot_surfaceome_gene_name.tab'), sep = '\t')

In [ ]:
surfaceome['ID link'] = surfaceome.apply(lambda row: row['ID link'].split(','), axis = 1)
surfaceome = surfaceome.explode('ID link')

In [ ]:
expression = pd.read_csv(resolve('inputs/expression/E-PROT-1-query-results.tsv'), sep = '\t',
                        header = 4)

surf = pd.merge(surfaceome, expression, left_on = 'Gene names  (primary )', right_on = 'Gene Name', how = 'inner')

In [ ]:
heights = pd.read_csv(resolve('database/height_estimates.csv'))

heights.drop(columns = ['index'], axis =1, inplace = True)
heights.rename(columns = {"ID link_first":"ID link",
                         "Entry name_first": "Entry name",
                         "dom_len_count_domain": "domain counts"}, inplace = True)

surf_h = pd.merge(surf.drop_duplicates(subset = ['Gene ID']), heights, on = 'ID link')

In [ ]:
#The surface of each immune cell type: the surfaceome proteins it expresses, each with its
#height and its share of the cell's total surface expression (percent_expression).
cell_types = {
    'bcell': 'adult, B cell',
    'cd4tcell': 'adult, CD4-positive T cell',
    'cd8tcell': 'adult, CD8-positive T cell',
    'monocyte': 'adult, monocyte',
    'nk': 'adult, natural killer cell',
    'platelet': 'adult, platelet',
}
surfaces = {name: cell_surface(surf_h, column, columns=['ID link', column, 'total_height', 'Gene Name'])
            for name, column in cell_types.items()}

#expression-weighted mean height of each surface, nm
pd.Series({name: mean_height(surface) for name, surface in surfaces.items()}).round(2)

In [ ]:
surfaces['cd4tcell'].to_csv(out_path('tables/expression_eprot1_cd4tcell.csv'))
surfaces['bcell'].to_csv(out_path('tables/expression_eprot1_bcell.csv'))
surfaces['nk'].to_csv(out_path('tables/expression_eprot1_nkcell.csv'))
surfaces['monocyte'].to_csv(out_path('tables/expression_eprot1_monocyte.csv'))
surfaces['platelet'].to_csv(out_path('tables/expression_eprot1_platelet.csv'))
surfaces['cd8tcell'].to_csv(out_path('tables/expression_eprot1_cd8tcell.csv'))

In [ ]:
surfaceome.to_csv(out_path('tables/surfaceome_ids_mapped.csv'))

In [ ]:
#Figures 3-4: each immune cell's surface against the CD4+ T cell's
import matplotlib
matplotlib.rcParams['pdf.fonttype'] = 42
matplotlib.rcParams['ps.fonttype'] = 42

cd4 = {'CD4+ T-cell': surfaces['cd4tcell']}
plot_surfaces({**cd4, 'B Cell': surfaces['bcell']}).savefig(out_path('figures/F3_tcell_vs_bcell.pdf'), transparent=True)
plot_surfaces({**cd4, 'NK Cell': surfaces['nk']}).savefig(out_path('figures/F3_tcell_vs_nkcell.pdf'), transparent=True)
plot_surfaces({**cd4, 'Monocyte Cell': surfaces['monocyte']}).savefig(out_path('figures/F3_tcell_vs_monocyte.pdf'), transparent=True)
plot_surfaces({**cd4, 'CD8 T Cell': surfaces['cd8tcell']}).savefig(out_path('figures/F3_tcell_vs_cd8tcell.pdf'), transparent=True)
plot_surfaces({**cd4, 'platelet': surfaces['platelet']}).savefig(out_path('figures/F3_tcell_vs_platelet.pdf'), transparent=True)